<div dir="rtl">

# رافِدَين — تدريب Seed-VC على اللهجة العراقية على منصة RunPod

دفتر Jupyter واحد متكامل ينجز **الرحلة كاملة** على حاوية GPU من RunPod — من بيئة فارغة إلى نموذج
`seed-uvit-tat-xlsr-tiny` مضبوط بدقة على الكلام العراقي، محفوظاً بشكل **دائم** يمكنك تنزيله متى شئت:

| المرحلة | ماذا يحدث |
|---|---|
| 1. البيئة | فحص GPU/القرص، تثبيت ffmpeg وuv وPyTorch 2.4.0 (cu121) في بيئة معزولة قابلة لإعادة الاستخدام |
| 2. المستودعات | استنساخ Seed-VC ورافِدَين، واستخراج خط بيانات العراق المدمج (49,194 مقطعاً / 339 ساعة) |
| 3. البيانات | تنزيل 3 مجموعات عراقية نظيفة + جلب مُجزَّأ (ranged HTTP) لساعات YODAS العراقية، ثم تقطيع وترميز Opus 24kHz |
| 4. التدريب | ضبط دقيق يبدأ من نقطة `DiT_uvit_tat_xlsr_ema.pth` الرسمية، بتقدم حي ومنحنى خسارة |
| 5. التحقق والتصدير | استدلال حقيقي بالنقطة الناتجة، ثم حزمة تصدير (أوزان + إعداد + تقرير + سجلات) |
| 6. الحفظ الدائم | نسخة على القرص الشبكي الدائم + أرشيف tar.gz + **رفع إلى HuggingFace Hub** (دائم فعلاً) |

> هذا الدفتر مبني مباشرة على خط التدريب الرسمي للمستودع (`seed_vc_iraqi_ubuntu.sh`) — نفس المراحل
> ونفس الإصدارات المثبتة — لكنه مقسّم إلى خلايا قابلة للتشغيل والمراقبة والاستئناف، تماماً كتجربة
> Google Colab، مع تخزين دائم يناسب RunPod. كل خلية آمنة لإعادة التشغيل: المراحل تتخطى ما أُنجز
> منها وتستأنف ما بقي.

**المصدر**: [github.com/Alqudimi/rafidain](https://github.com/Alqudimi/rafidain) ·
النموذج: [Plachta/Seed-VC](https://huggingface.co/Plachta/Seed-VC)‏ (`seed-uvit-tat-xlsr-tiny`)

</div>

<div dir="rtl">

## ١) تجهيز الحاوية على RunPod (قبل فتح الدفتر)

### أ. إنشاء الحاوية (Pod)
1. من [runpod.io](https://runpod.io) → **GPU Pods** → **Deploy**.
2. اختر GPU من الجدول أدناه — كل ما يملك 16GB ذاكرة أو أكثر يعمل، والدفتر يضبط حجم الدفعة تلقائياً.
3. **القالب (Template)**: أي قالب PyTorch رسمي من RunPod (مثل *RunPod Pytorch 2.x*) أو أي قالب
   CUDA 12.x — الدفتر يبني بيئته الخاصة بالكامل ولا يحتاج من القالب سوى مشغل NVIDIA وJupyter.
4. فعّل خيار **Start Jupyter Notebook** — سيظهر زر *Connect to Jupyter Lab* بعد إقلاع الحاوية.
5. **التخزين — أهم نقطة**: أضف **Network Volume** (قرص شبكي دائم) بحجم **50GB أو أكثر**؛ يُوصل
   تلقائياً عند المسار `/workspace` ويبقى محفوظاً حتى بعد إيقاف أو **حذف** الحاوية. بدون قرص شبكي
   سيعمل الدفتر على قرص الحاوية المؤقت مع تحذير واضح في خلية الفحص — وذلك القرص يضيع عند حذف الحاوية.

| GPU | الذاكرة | حجم الدفعة المقترح | ملاحظات |
|---|---|---|---|
| RTX 3090 / 4090 | 24GB | 4 | أفضل نسبة سرعة/سعر لهذا التدريب |
| A40 / RTX A6000 | 48GB | 6 | أسرع، مناسب للبيانات الكبيرة |
| A100 40/80GB | 40–80GB | 6–8 | فائض عن حاجة نموذج 25M باراميتر |
| T4 / أي 16GB | 16GB | 2 | يعمل لكنه أبطأ — دليل المستودع يقدّر 1500–4000 خطوة بـ 1–4 ساعات على A10G/T4 |

### ب. حجم القرص الشبكي — تقديرات مقاسة فعلياً من كتالوج البيانات المدمج
| الإعداد | تنزيل YODAS الخام | الإجمالي التقريبي (بيانات + بيئة + نقاط) | القرص المقترح |
|---|---|---|---|
| 50 ساعة | ≈ 3.1GB | ≈ 12GB | 25GB |
| **200 ساعة (الافتراضي)** | **≈ 12.7GB** | **≈ 30GB** | **50GB** |
| 334 ساعة (الكتالوج كله) | ≈ 22.4GB | ≈ 42GB | 100GB |

### ج. نوع التشغيل
- **On-Demand**: أغلى قليلاً، بلا انقطاع — الأسهل لأول تجربة.
- **Spot / Interruptible**: أرخص كثيراً ويصلح هنا لأن **كل مرحلة قابلة للاستئناف** — بعد أي انقطاع أعد
  تشغيل الخلايا بالترتيب: البيانات تُتخطى إن كانت منجزاً (علامات `.done`)، والتدريب يستأنف من آخر
  نقطة حفظ تلقائياً.

### د. فتح الدفتر
بعد إقلاع الحاوية: **Connect → Connect to Jupyter Lab**، ثم اسحب ملف `rafidain_runpod_training.ipynb`
هذا إلى مجلد `/workspace` في متصفح الملفات (سحب وإفلات)، افتحه، واختر نواة Python 3، وشغّل الخلايا
من الأعلى إلى الأسفل.

</div>

<div dir="rtl">

## ٢) خريطة الدفتر — شغّل بالترتيب من الأعلى

| # | الخلية | الوصف | المدة التقريبية |
|---|---|---|---|
| 1 | الإعدادات المركزية | كل مقابض التحكم + الأدوات — تُحفظ على القرص الدائم | ثوانٍ |
| 2 | فحص البيئة | GPU/قرص/أدوات + تحديد حجم الدفعة تلقائياً | ثوانٍ |
| 3 | تبعيات النظام | ffmpeg + git + uv | أقل من دقيقة |
| 4 | بيئة Python | venv مستقل + PyTorch 2.4.0 cu121 + الاعتماديات المثبتة | 3–8 دقائق |
| 5 | المستودعات | استنساخ Seed-VC + رافِدَين | أقل من دقيقة |
| 6 | خط البيانات | استخراج خط بيانات العراق المدمج + إحصاءات الكتالوج | ثوانٍ |
| 7 | المجموعات النظيفة | youssef + shiekh + safaa (نحو 3GB) | 2–10 دقائق |
| 8 | بيانات YODAS | جلب مُجزَّأ بحسب الساعات المطلوبة (12.7GB لـ 200 ساعة) | 15–60 دقيقة |
| 9 | بياناتك الخاصة | (اختياري) مجلد لملفاتك الصوتية يُدمج في التدريب | حسبك |
| 10 | التقطيع والترميز | تقطيع 1–29.5 ثانية وترميز Opus 24kHz أحادي | 5–20 دقيقة |
| 11 | نقاط النموذج | تنزيل DiT/XLSR/HiFT/CAMPPlus مسبقاً (نحو 2.2GB) | 2–10 دقائق |
| 12 | إطلاق التدريب | عملية خلفية تنجو من إغلاق المتصفح وإعادة تشغيل النواة | فوري |
| 13 | مراقبة التدريب | تقدم + ETA + منحنى الخسارة — أعد تشغيلها متى شئت | حسب الرغبة |
| 14 | انتظار الاكتمال | (اختياري) مراقبة تلقائية حتى النهاية | مدة التدريب |
| 15 | التحقق | استدلال حقيقي + استماع داخل الدفتر | 1–3 دقائق |
| 16 | التصدير | حزمة: أوزان + إعداد + تقرير + سجلات | ثوانٍ |
| 17 | أرشيف tar.gz | ملف واحد جاهز للتنزيل من متصفح Jupyter | ثوانٍ |
| 18 | HuggingFace Hub | **التخزين السحابي الدائم** — النموذج في حسابك للأبد | 1–5 دقائق |
| 19–21 | إضافية | rclone سحابي / تنظيف مساحة / دليل النشر في خدمة رافِدَين | اختيارية |

**قواعد ذهبية**
- كل الخلايا **آمنة لإعادة التشغيل** — لا شيء يُعاد تنفيذه إن كان منجزاً (علامات `.done` وفحوص وجود).
- إذا انقطع كل شيء (Spot أو إعادة تشغيل): افتح الدفتر وشغّل الخلايا **بالترتيب** من جديد — سيستأنف تلقائياً.
- التدريب وحده يعمل كعملية خلفية مستقلة عن النواة؛ راقبه بالخلية 13.
- لا تحذف الـ **Network Volume** من لوحة RunPod — فيه كل مقتنياتك الدائمة.

</div>

In [ ]:
# ============================================================ #
#  الخلية ١ — الإعدادات المركزية والأدوات المساعدة              #
#  كل مقابض التحكم في مكان واحد، وتُحفظ على القرص الدائم        #
#  (إعادة تشغيل النواة لا تفقدها — وتُعاد عند الاستئناف)         #
# ============================================================ #
import os, sys, json, subprocess, shutil, time, re
from pathlib import Path

# ---------- 1) جذر العمل: على تخزين دائم إن وُجد ----------
# على RunPod: القرص الشبكي (Network Volume) يُوصل تلقائياً في /workspace وهو دائم.
def _pick_root():
    # IRAQI_DATA_ROOT = مسار جذر العمل كاملاً مباشرة (نفس دلالته في سكربتات المستودع)
    override = os.environ.get("IRAQI_DATA_ROOT", "").strip()
    if override:
        return Path(override)
    for cand in ("/workspace", "/root"):
        p = Path(cand)
        try:
            p.mkdir(parents=True, exist_ok=True)
            (p / ".write_test").write_text("ok"); (p / ".write_test").unlink()
            return p / "iraqi_seedvc"
        except Exception:
            continue
    return Path.home() / "iraqi_seedvc"

ROOT = _pick_root()   # /workspace/iraqi_seedvc (دائم) أو /root/iraqi_seedvc (مؤقت)
ROOT.mkdir(parents=True, exist_ok=True)
CFG_PATH  = ROOT / "nb_config.json"

# ---------- 2) المقابض الافتراضية ----------
DEFAULTS = dict(
    # البيانات
    YODAS_HOURS    = 200,    # ساعات YODAS العراقية (0 = تعطيل، "all" = الكل 334 ساعة، أو أي رقم)
    SKIP_SMALL     = False,  # تخطي المجموعات النظيفة الصغيرة الثلاث
    # التدريب
    RUN_NAME       = "iraqi_v1",  # اسم التشغيل (مجلد النقاط والحزمة)
    STEPS          = 3000,   # عدد خطوات الضبط الدقيق (500 لمتحدث واحد، 1500-4000 للهجة عموماً)
    BATCH          = "auto", # حجم الدفعة: "auto" = حسب ذاكرة GPU، أو رقم 1/2/4/6/8
    SAVE_EVERY     = 250,    # حفظ نقطة تفتيش كل N خطوة (يُحتفظ بآخر نقطتين)
    TRAIN_WORKERS  = 2,      # عمال تحميل بيانات التدريب
    # التنزيل
    FETCH_WORKERS  = 16,     # تنزيلات YODAS المتوازية (ارفعها إلى 32 إذا كانت الشبكة سريعة)
    PREP_WORKERS   = 0,      # عمال التقطيع والترميز (0 = تلقائي: حتى 16 نواة)
    HF_ENDPOINT    = "",     # مرآة HuggingFace إن لزم (مثال: https://hf-mirror.com) — اتركها فارغة عادة
    # التخزين الدائم (HuggingFace Hub)
    HF_TOKEN       = "",     # رمز من hf.co/settings/tokens بصلاحية Write — للرفع الدائم (الخلية 18)
    HF_REPO_ID     = "",     # فارغ = <اسم-حسابك>/rafidain-seedvc-iraqi تلقائياً
    HF_PRIVATE     = True,   # مستودع خاص لا يراه أحد غيرك
    # تنظيف
    CLEAN_RAW_AFTER = False, # حذف المصادر الخام بعد تجهيز بيانات التدريب (يوفر مساحة كبيرة — الخلية 20)
)

# ---------- 3) إعادة تحميل الإعدادات المحفوظة (استئناف بعد إعادة تشغيل النواة) ----------
CFG = dict(DEFAULTS)
FORCE_RESET = False   # اجعلها True مؤقتاً لإعادة كل الإعدادات إلى الافتراضية
if CFG_PATH.exists() and not FORCE_RESET:
    try:
        saved = json.loads(CFG_PATH.read_text(encoding="utf-8"))
        CFG.update({k: saved[k] for k in DEFAULTS if k in saved})
        print(f"[إعدادات] عيد تحميلها من {CFG_PATH} (عدّل ثم أعد التشغيل، أو FORCE_RESET=True للبدء من جديد)")
    except Exception as e:
        print(f"[إعدادات] تعذر قراءة المحفوظة ({e}) — نستخدم الافتراضية")
# ← عدّل أي قيمة هنا ثم أعد تشغيل الخلية، مثال:
#    CFG["YODAS_HOURS"] = 50
#    CFG["STEPS"] = 1000
CFG_PATH.write_text(json.dumps(CFG, ensure_ascii=False, indent=2), encoding="utf-8")

# ---------- 4) المسارات الثابتة (كلها تحت الجذر الدائم) ----------
SEEDVC   = ROOT / "seed-vc"                    # مستودع Seed-VC (المدرب الأصلي)
RAFIDAIN = ROOT / "rafidain"                   # مستودع رافِدَين (الخدمة + خط البيانات)
PIPE     = ROOT / "iraqi_pipeline.py"          # خط بيانات العراق — يُستخرج في الخلية 6
DATA_DIR = ROOT / "data" / "iraqi_finetune"    # بيانات التدريب المجهزة (Opus 24kHz)
RUN_DIR  = SEEDVC / "runs" / CFG["RUN_NAME"]   # نقاط التدريب والسجلات
EXPORTS  = ROOT / "exports"                    # حزم النموذج النهائية
VENV_PY  = ROOT / "venv" / "bin" / "python"    # بايثون البيئة المعزولة (الخلية 4)
for p in (ROOT, EXPORTS, RUN_DIR):
    p.mkdir(parents=True, exist_ok=True)

# ---------- 5) بيئة الأوامر وأداة تشغيل موحدة ----------
ENV = dict(os.environ)
ENV["PYTHONUNBUFFERED"] = "1"
ENV["HF_HUB_DISABLE_XET"] = "1"           # تنزيل HTTP قابل للاستئناف (نفس خيار سكربتات المستودع)
ENV["TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD"] = "1"
if CFG["HF_ENDPOINT"]:
    ENV["HF_ENDPOINT"] = CFG["HF_ENDPOINT"]

def sh(cmd, cwd=None, env=None, check=True, echo=True):
    # تشغيل أمر وطباعة مخرجاته لحظياً داخل الخلية — يعيد رمز الخروج
    cmd = [str(c) for c in cmd]
    if echo:
        print("$ " + " ".join(cmd), flush=True)
    proc = subprocess.Popen(cmd, cwd=str(cwd) if cwd else None,
                            env=env or ENV, text=True, bufsize=1,
                            stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    for line in proc.stdout:
        print(line.rstrip(), flush=True)
    rc = proc.wait()
    if check and rc != 0:
        raise RuntimeError(f"فشل الأمر (رمز {rc}): {' '.join(cmd)}")
    return rc

# ---------- 6) ملخص ----------
print("=" * 64)
print(f"الجذر الدائم للعمل : {ROOT}")
print("=" * 64)
print(f"  • ساعات YODAS        : {CFG['YODAS_HOURS']}")
print(f"  • التدريب            : {CFG['STEPS']} خطوة، حفظ كل {CFG['SAVE_EVERY']}, دفعة {CFG['BATCH']}")
print(f"  • اسم التشغيل        : {CFG['RUN_NAME']}")
print(f"  • رمز HuggingFace    : {'مضبوط (سيُرفع للأبد في الخلية 18)' if CFG['HF_TOKEN'] else 'غير مضبوط (اختياري — الرفع الدائم)'}")
print(f"\n[المسارات]  البيانات: {DATA_DIR}")
print(f"            التدريب : {RUN_DIR}")
print(f"            التصدير : {EXPORTS}")
print(f"\nملاحظة: الرمز السري (إن وضعته) يُحفظ في {CFG_PATH} على قرصك الدائم فقط.")

In [ ]:
# ============================================================ #
#  الخلية ٢ — فحص البيئة: GPU / قرص / أدوات                    #
#  (يحدد حجم الدفعة تلقائياً ويكشف مدى ديمومة التخزين)          #
# ============================================================ #
import subprocess, os, shutil, re, json
from pathlib import Path

def gpu_query():
    try:
        out = subprocess.run(
            ["nvidia-smi", "--query-gpu=name,memory.total,driver_version",
             "--format=csv,noheader,nounits"],
            capture_output=True, text=True, timeout=30).stdout.strip().splitlines()
        return [l.strip() for l in out if l.strip()]
    except Exception:
        return []

gpus = gpu_query()
if gpus:
    for i, g in enumerate(gpus):
        parts = [p.strip() for p in g.split(",")]
        name, vram_mib, driver = parts[0], float(parts[1]), parts[2]
        print(f"[GPU {i}] {name} — ذاكرة {vram_mib/1024:.1f}GB — مشغل {driver}")
else:
    print("[GPU] لا يوجد GPU ظاهر عبر nvidia-smi — لن يعمل التدريب (تجهيز البيانات فقط).")

def auto_batch(vram_gb):
    if vram_gb >= 70: return 8
    if vram_gb >= 40: return 6
    if vram_gb >= 20: return 4
    if vram_gb >= 15: return 2
    return 1

if gpus:
    vram_gb = float([p.strip() for p in gpus[0].split(",")][1]) / 1024
    CFG["_BATCH"] = auto_batch(vram_gb)
else:
    vram_gb = 0.0
    CFG["_BATCH"] = 2
if str(CFG["BATCH"]).lower() == "auto":
    BATCH = CFG["_BATCH"]
    print(f"[دفعة] تلقائي => BATCH = {BATCH} (حسب {vram_gb:.0f}GB ذاكرة)")
else:
    BATCH = int(CFG["BATCH"])
    print(f"[دفعة] يدوي  => BATCH = {BATCH}")
CFG["_BATCH"] = BATCH
CFG_PATH.write_text(json.dumps(CFG, ensure_ascii=False, indent=2), encoding="utf-8")

# ---- المعالج والذاكرة ----
cpu = os.cpu_count() or 4
try:
    mem_kb = int(re.search(r"MemTotal:\s+(\d+)", Path("/proc/meminfo").read_text()).group(1))
    print(f"[نظام] {cpu} نواة معالج — ذاكرة RAM ≈ {mem_kb/1e6:.1f}GB")
except Exception:
    print(f"[نظام] {cpu} نواة معالج")

# ---- القرص ونوع التخزين (ديمومة الملفات!) ----
du = shutil.disk_usage(ROOT)
print(f"[قرص] المتاح على {ROOT}: حر {du.free/1e9:.1f}GB من {du.total/1e9:.1f}GB")
def find_mount(path):
    best, fstype, dev = "", "", ""
    try:
        for line in Path("/proc/mounts").read_text().splitlines():
            parts = line.split()
            if len(parts) >= 3 and str(path).startswith(parts[1]) and len(parts[1]) > len(best):
                best, fstype, dev = parts[1], parts[2], parts[0]
    except Exception:
        pass
    return best, fstype, dev
mnt, fstype, dev = find_mount(ROOT)
print(f"[تخزين] نقطة الوصل: {mnt or '/'} — النوع: {fstype or 'غير معروف'} — الجهاز: {dev or '?'}")
if str(ROOT).startswith("/workspace"):
    print("        [جيد] العمل تحت /workspace — إن كان هذا القرص الشبكي (Network Volume) فملفاتك")
    print("               دائمة وتبقى بعد إيقاف/حذف الحاوية (لا تحذف الـ Volume من لوحة RunPod).")
elif str(ROOT).startswith("/root") or str(ROOT).startswith("/home"):
    print("        [تحذير] لا يوجد /workspace — أنت على قرص الحاوية المحلي:")
    print("                يبقى عند الإيقاف (Stop) لكنه يضيع عند الحذف (Terminate).")
    print("                لضمان الديمومة: أنشئ Network Volume في RunPod وأعد تشغيل الدفتر،")
    print("                أو اضبط IRAQI_DATA_ROOT، أو اعتمد على الرفع الدائم (الخلية 18).")

# ---- الأدوات ----
for tool in ("ffmpeg", "ffprobe", "git", "curl", "uv"):
    print(f"[أداة] {tool:<8}: {'متوفر' if shutil.which(tool) else 'غير متوفر'}")

print("\n[فحص مبدئي] أكمل إلى الخلية 3 (التبعيات).")

In [ ]:
# ============================================================ #
#  الخلية ٣ — تبعيات النظام (ffmpeg / git) + مدير الحزم uv     #
#  (حاويات RunPod تعمل بصلاحيات root عادة — بلا sudo)          #
# ============================================================ #
import shutil, os, subprocess
from pathlib import Path

def have(x):
    return shutil.which(x) is not None

missing = [p for p in ("ffmpeg", "git") if not have(p)]
if missing:
    print(f"تثبيت حزم النظام: {', '.join(missing)} ...")
    sh(["apt-get", "update", "-qq"], check=False)
    sh(["apt-get", "install", "-y", "-qq", *missing])
for tool in ("ffmpeg", "git", "curl"):
    print(f"  {tool:<8}: {'متوفر' if have(tool) else 'غير متوفر!'}")

# ---- uv: مُسرّع تثبيت بايثون (نفس مسار سكربتات المستودع) ----
UV = shutil.which("uv")
if UV is None:
    print("\nتثبيت uv (مرة واحدة) ...")
    sh(["bash", "-c", "curl -LsSf https://astral.sh/uv/install.sh | sh"], check=False)
    for cand in (Path.home() / ".local" / "bin" / "uv", Path.home() / ".cargo" / "bin" / "uv"):
        if cand.exists():
            UV = str(cand)
            break
    if UV:
        os.environ["PATH"] = str(Path(UV).parent) + os.pathsep + os.environ["PATH"]
        ENV["PATH"] = os.environ["PATH"]
if UV:
    r = subprocess.run([UV, "--version"], capture_output=True, text=True)
    print(f"\nuv جاهز: {r.stdout.strip() or r.stderr.strip()}")
else:
    print("\nuv غير متوفر — سنعتمد python3 -m venv + pip كبديل في الخلية 4 (يعمل أيضاً).")

In [ ]:
# ============================================================ #
#  الخلية ٤ — بيئة Python المعزولة + PyTorch 2.4.0 (CUDA 12.1) #
#  نفس الإصدارات المثبتة في سكربتات المستودع الرسمية            #
#  (تُبنى مرة واحدة على القرص الدائم وتُعاد استخدامها دائماً)    #
# ============================================================ #
import subprocess, shutil, sys
from pathlib import Path

UV = globals().get("UV") or shutil.which("uv")   # تحصين: يعمل حتى بعد إعادة تشغيل النواة

def venv_ok(code):
    # فحص سريع داخل البيئة المعزولة
    if not VENV_PY.exists():
        return False
    r = subprocess.run([str(VENV_PY), "-c", code], capture_output=True, text=True, env=ENV)
    return r.returncode == 0

# ---- 1) إنشاء venv إن لم يوجد ----
if not VENV_PY.exists():
    print(f"إنشاء بيئة معزولة (Python 3.11) في {ROOT/'venv'} ...")
    if UV:
        sh([UV, "venv", "--python", "3.11", str(ROOT / "venv")])
    else:
        sh([sys.executable, "-m", "venv", str(ROOT / "venv")])

# ---- 2) PyTorch 2.4.0 cu121 (نحو 2.5GB — يُتخطى إن كان مثبتاً) ----
if not venv_ok("import torch; assert torch.__version__.startswith('2.4.')"):
    print("تثبيت PyTorch 2.4.0 + torchaudio (cu121) — عدة دقائق حسب الشبكة ...")
    trio_pin = ["torch==2.4.0", "torchvision==0.19.0", "torchaudio==2.4.0"]
    trio_any = ["torch", "torchvision", "torchaudio"]
    idx = ["--index-url", "https://download.pytorch.org/whl/cu121"]
    if UV:
        rc = sh([UV, "pip", "install", "--python", str(VENV_PY), *trio_pin, *idx], check=False)
        if rc != 0:
            sh([UV, "pip", "install", "--python", str(VENV_PY), *trio_any, *idx])
    else:
        rc = sh([str(VENV_PY), "-m", "pip", "install", "-q", *trio_pin, *idx], check=False)
        if rc != 0:
            sh([str(VENV_PY), "-m", "pip", "install", "-q", *trio_any, *idx])
else:
    print("PyTorch 2.4.x موجود أصلاً في البيئة — تخطّي التثبيت.")

# ---- 3) بقية الاعتماديات (مثبتة بالإصدارات نفسها التي نجح بها خط المستودع) ----
NEED = ("import torch, torchaudio, librosa, soundfile, munch, einops,"
        " transformers, pyarrow, huggingface_hub, yaml, matplotlib")
if not venv_ok(NEED):
    print("تثبيت الاعتماديات (numpy/scipy/librosa/transformers/...) ...")
    pins = ["numpy==1.26.4", "scipy==1.13.1", "librosa==0.10.2", "soundfile==0.12.1",
            "munch==4.0.0", "einops==0.8.0", "transformers==4.46.3", "pyyaml", "tqdm",
            "requests", "pyarrow==17.0.0", "huggingface_hub", "matplotlib"]
    loose = ["numpy", "scipy", "librosa", "soundfile", "munch", "einops", "transformers",
             "pyyaml", "tqdm", "requests", "pyarrow", "huggingface_hub", "matplotlib"]
    if UV:
        rc = sh([UV, "pip", "install", "--python", str(VENV_PY), *pins], check=False)
        if rc != 0:
            sh([UV, "pip", "install", "--python", str(VENV_PY), *loose])
    else:
        rc = sh([str(VENV_PY), "-m", "pip", "install", "-q", *pins], check=False)
        if rc != 0:
            sh([str(VENV_PY), "-m", "pip", "install", "-q", *loose])
else:
    print("الاعتماديات موجودة أصلاً — تخطّي التثبيت.")

# ---- 4) تحقق نهائي ----
r = subprocess.run([str(VENV_PY), "-c",
                    "import torch, torchaudio; print('torch', torch.__version__, '| cuda:', torch.cuda.is_available())"],
                   capture_output=True, text=True, env=ENV)
print((r.stdout + r.stderr).strip())
assert r.returncode == 0 and "torch 2.4" in (r.stdout + r.stderr), "فشل بناء البيئة — راجع السجل أعلاه"
print(f"\n[جاهز] البيئة المعزولة: {VENV_PY}")

In [ ]:
# ============================================================ #
#  الخلية ٥ — استنساخ المستودعات: Seed-VC + رافِدَين            #
#  (على القرص الدائم — يُحدثان إن وُجدا بدلاً من إعادة التنزيل) #
# ============================================================ #
from pathlib import Path

def git_clone(url, dst):
    if (dst / ".git").exists():
        print(f"تحديث {dst.name} ...")
        sh(["git", "-C", str(dst), "pull", "--ff-only"], check=False)
    else:
        print(f"استنساخ {dst.name} من {url} ...")
        sh(["git", "clone", "--depth", "1", url, str(dst)])

git_clone("https://github.com/Plachtaa/seed-vc.git", SEEDVC)
git_clone("https://github.com/Alqudimi/rafidain.git", RAFIDAIN)

# تحقق من الملفات الحرجة
for f in ("train.py", "inference.py", "configs/presets/config_dit_mel_seed_uvit_xlsr_tiny.yml"):
    assert (SEEDVC / f).exists(), f"ملف مفقود في seed-vc: {f}"
assert (RAFIDAIN / "seed_vc_iraqi_ubuntu.sh").exists(), "سكربت خط البيانات مفقود في rafidain"
print(f"\n[جاهز] seed-vc   : {SEEDVC}")
print(f"[جاهز] rafidain  : {RAFIDAIN}")

In [ ]:
# ============================================================ #
#  الخلية ٦ — استخراج خط بيانات العراق المدمج                   #
#  (الكتالوج العراقي كله مضمّن داخل سكربت المستودع نفسه:         #
#   49,194 مقطعاً / 339 ساعة / خريطة إزاحات بايت-دقيقة           #
#   داخل أرشيفات espnet/yodas3 — نستخرجه هنا كوحدة مستقلة)      #
# ============================================================ #
import re, gzip, base64, json, ast, urllib.request
from pathlib import Path

SRC_SH = RAFIDAIN / "seed_vc_iraqi_ubuntu.sh"
if not SRC_SH.exists():   # احتياط: المستودع غير منسوخ — ننزّل السكربت مباشرة من GitHub
    print("تنزيل السكربت من GitHub مباشرة ...")
    urllib.request.urlretrieve(
        "https://raw.githubusercontent.com/Alqudimi/rafidain/main/seed_vc_iraqi_ubuntu.sh", SRC_SH)

text = SRC_SH.read_text(encoding="utf-8")
m = re.search(r"cat > \"\$ROOT/iraqi_pipeline\.py\" <<'PYEOF'\n(.*?)\nPYEOF\n", text, re.S)
assert m, "لم يُعثر على خط البيانات المدمج داخل السكربت!"
code = m.group(1)
ast.parse(code)   # تحقق من سلامة الكود المستخرج قبل الكتابة
PIPE.write_text(code, encoding="utf-8")
print(f"[خط البيانات] استُخرج إلى: {PIPE}  ({len(code):,} بايت)")

# ---- إحصاءات الكتالوج العراقي المدمج (بدون أي تنزيل) ----
def _grab(name):
    blob = re.search(name + r' = r"""(.*?)"""', code, re.S).group(1)
    return json.loads(gzip.decompress(base64.b64decode(blob)))

clips, offs, srcrank = _grab("_IQ_CLIPS_B64"), _grab("_IQ_OFFS_B64"), _grab("_IQ_SRC_B64")
hours_total = sum(c["du"] for c in clips) / 3600
resolvable = [s for s, v in offs.items() if v.get("off")]
print(f"[كتالوج اللهجة العراقية] {len(clips):,} مقطعاً | {hours_total:.0f} ساعة | "
      f"{len(resolvable):,} تسجيل مصدر قابل للجلب | {sum(offs[s]['size'] for s in resolvable)/1e9:.1f}GB لو جُلب كله")

# ---- حجم هدفك الحالي (نفس منطق الاختيار في السكربت: الجودة أولاً) ----
t = str(CFG["YODAS_HOURS"]).strip().lower()
if t in ("0", ""):
    print(f"\n[هدفك] YODAS معطّل (YODAS_HOURS=0) — الخلية 8 ستتخطاه. الأصل المتاح: {hours_total:.0f} ساعة.")
else:
    target = None if t in ("all", "كل") else float(t)
    ranked = sorted(srcrank.items(), key=lambda kv: -kv[1][2])
    chosen, h = [], 0.0
    for sid, (tar, ch, q) in ranked:
        if sid not in offs or offs[sid].get("off") is None:
            continue
        chosen.append(sid); h += ch
        if target and h >= target:
            break
    size_gb = sum(offs[s]["size"] for s in chosen) / 1e9
    label = f"{target:.0f} ساعة" if target else "الكتالوج كله"
    print(f"[هدفك] {label}: {len(chosen):,} تسجيلاً مصدراً ≈ {size_gb:.1f}GB تنزيلاً خاماً (سيُشفَّر إلى Opus للتدريب)")
print("\n[جاهز] تابع إلى خلايا البيانات (7 وما بعدها).")

<div dir="rtl">

## مرحلة البيانات — المصادر

| المصدر | ما هو | الحجم | ملاحظات |
|---|---|---|---|
| `youssef` | 3,867 مقطعاً عراقياً 22.05kHz مع نصوص | ≈ 720MB | مقاطع قصيرة نظيفة |
| `shiekh` | 827 مقطعاً 16kHz — متحدث واحد | ≈ 640MB | يساعد ثبات المتحدث الواحد |
| `safaa` | 676 ملف wav — متحدثة أنثى جاهزة لـTTS | مئات MB | نظيفة جداً |
| YODAS‑IQ | مقاطع عراقية مصنّفة من كتالوج YODAS3 (49,194 مقطعاً / 339 ساعة / 2,360 تسجيلاً) | حسب ساعاتك | تُجلب **بتنزيل مُجزَّأ**: نطاق البايتات المطلوب فقط من ملفات tar الضخمة — دون تنزيل الأرشيف كله (لهذا 200 ساعة = 12.7GB فقط بدل 22GB+) |
| `custom` | **ملفاتك أنت** — عبر الخلية 9 | حسبك | تسجيلات بموافقة أصحابها، نظيفة قدر الإمكان |

المرحلة 10 (التقطيع والترميز) تحوّل كل ما سبق إلى مقاطع **Opus 24kHz أحادية بطول 1–29.5 ثانية** —
الشكل الذي يقرأه مدرب Seed-VC مباشرة (يفحص المجلد بشكل تكراري: `yodas/`, `youssef/`, `custom/` ...).

فشل أحد المصادر الصغيرة ليس قاتلاً — الخط يتابع بالبقية ويمكن إعادة المحاولة لاحقاً (كل المصادر
تُعلَّم بعلامات `.done` عند الإنجاز فلا يُعاد تنزيلها).

</div>

In [ ]:
# ============================================================ #
#  الخلية ٧ — تنزيل المجموعات العراقية النظيفة الصغيرة          #
#  (youssef / shiekh / safaa من HuggingFace — ≈2-3GB)          #
#  قابلة للاستئناف: أعد التشغيل لتكملة ما فشل فقط              #
# ============================================================ #
if CFG["SKIP_SMALL"]:
    print("تخطي المجموعات الصغيرة (SKIP_SMALL=True) — إن أردت فعّلها من الخلية 1.")
else:
    sh([VENV_PY, PIPE, "small", "--root", ROOT])
    import os
    for name in ("youssef", "shiekh", "safaa"):
        d = ROOT / "raw" / name
        n = len(list(d.glob("*"))) if d.exists() else 0
        print(f"  {name:<8}: {n:,} ملفاً في {d}")
print("\n[جاهز] تابع إلى الخلية 8 (YODAS) — أو الخلية 9 لدمج ملفاتك الخاصة.")

In [ ]:
# ============================================================ #
#  الخلية ٨ — جلب بيانات YODAS العراقية (الكتلة الكبرى)        #
#  تنزيل مُجزَّأ متوازٍ من أرشيفات espnet/yodas3                #
#  يستغرق عادة 15-60 دقيقة (حسب الشبكة وساعاتك)                #
#  قابلاً للاستئناف 100%: أعد التشغيل ليكمل من حيث توقف         #
# ============================================================ #
import shutil

t = str(CFG["YODAS_HOURS"]).strip().lower()
if t == "0":
    print("YODAS معطّل (YODAS_HOURS=0) — تخطّي.")
else:
    free0 = shutil.disk_usage(ROOT).free / 1e9
    print(f"المساحة الحرة قبل الجلب: {free0:.1f}GB")
    sh([VENV_PY, PIPE, "yodas",
        "--root", ROOT,
        "--hours", str(CFG["YODAS_HOURS"]),
        "--workers", int(CFG["FETCH_WORKERS"])])
    n = len(list((ROOT / "raw" / "yodas_webm").glob("*.webm"))) if (ROOT / "raw" / "yodas_webm").exists() else 0
    free1 = shutil.disk_usage(ROOT).free / 1e9
    print(f"\nتسجيلات مصدر جاهزة: {n:,} ملف webm | استُهلك {free0-free1:.1f}GB | الحر الآن {free1:.1f}GB")
    state = ROOT / "raw" / "yodas_fetch_state.json"
    if state.exists():
        import json
        failed = json.loads(state.read_text()).get("failed", [])
        if failed:
            print(f"مصادر فشل جلبها: {len(failed)} — أعد هذه الخلية لإعادة محاولة المتبقي، أو تجاهل (بقية البيانات تكفي)")
print("\n[جاهز] تابع إلى الخلية 9 (اختيارية) ثم 10 (التقطيع والترميز).")

In [ ]:
# ============================================================ #
#  الخلية ٩ — (اختيارية) دمج تسجيلاتك الخاصة في التدريب         #
#  ارفع ملفاتك (wav/flac/mp3/m4a/opus/ogg) إلى:                #
#     /workspace/iraqi_seedvc/raw/custom/                      #
#  عبر متصفح ملفات Jupyter (سحب وإفلات) ثم أعد تشغيل الخلية    #
#  ستُقطَّع تلقائياً إلى مقاطع 1-29.5 ثانية وتُرمَّز Opus 24kHz   #
# ============================================================ #
import sys, os, json, time
from multiprocessing import Pool
from pathlib import Path

CUSTOM = ROOT / "raw" / "custom"
CUSTOM.mkdir(parents=True, exist_ok=True)
EXTS = {".wav", ".mp3", ".flac", ".ogg", ".opus", ".m4a", ".webm"}
files = [p for p in sorted(CUSTOM.iterdir()) if p.is_file() and p.suffix.lower() in EXTS]
print(f"مجلد بياناتك: {CUSTOM}")
print(f"ملفات صوتية موجودة: {len(files)}")

if not files:
    print('''
لا توجد ملفات بعد — لتفعيل ذلك:
  1) ارفع تسجيلاتك إلى المجلد أعلاه (سحب وإفلات في لوحة ملفات Jupyter)
  2) الأفضل: كلام نظيف بلا موسيقى أو ضجيج، 1-30 ثانية أو أطول (تُقطَّع تلقائياً)
  3) أعد تشغيل هذه الخلية، ثم شغّل خلية التقطيع (10) لتشمل بياناتك
(تخطَّ هذه الخلية إن لم يكن لديك بيانات خاصة — مجموعة YODAS + الصغيرة كافية تماماً)''')
else:
    sys.path.insert(0, str(ROOT))
    import iraqi_pipeline as P   # خط البيانات المستخرج في الخلية 6 — نفس منطق التقطيع المجرَّب
    import types
    a = types.SimpleNamespace(out=str(DATA_DIR))
    jobs = []
    for f in files:
        jobs.extend(P._prep_job_src(a, str(f), "custom"))
    print(f"سيُنتج عنها {len(jobs)} مقطعاً (يُتخطى الموجود منها)")
    t0, ok_n, bad_n = time.time(), 0, 0
    with Pool(processes=int(CFG["PREP_WORKERS"] or 0) or min(16, os.cpu_count() or 4)) as pool:
        for out, dur, okk in pool.imap_unordered(P._run_prep_job, jobs, chunksize=16):
            if okk:
                ok_n += 1
            else:
                bad_n += 1
    print(f"تم: {ok_n} مقطعاً ناجحاً | {bad_n} فاشلاً في {time.time()-t0:.0f} ثانية")
    # تحديث فهرس بيانات التدريب (كما يفعل أمر prep نفسه)
    idx = DATA_DIR / "_prep_index.jsonl"
    prev = {}
    if idx.exists():
        for line in idx.read_text().splitlines():
            try:
                r = json.loads(line)
                prev[r["file"]] = r["dur"]
            except Exception:
                pass
    for j in jobs:
        prev[os.path.relpath(j[3], DATA_DIR)] = j[2]
    prev = {f: d for f, d in prev.items() if (DATA_DIR / f).exists()}
    DATA_DIR.mkdir(parents=True, exist_ok=True)
    idx.write_text("".join(json.dumps({"file": f, "dur": d}) + "\n" for f, d in sorted(prev.items())))
    print("فهرس بيانات التدريب محدّث — أكمل إلى الخلية 10.")

In [ ]:
# ============================================================ #
#  الخلية ١٠ — التقطيع والترميز: بناء بيانات التدريب النهائية   #
#  كل المقاطع -> Opus 24kHz أحادي، 1-29.5 ثانية                #
#  (يشمل YODAS + المجموعات الصغيرة + بياناتك إن وضعتها)        #
#  قابلة للاستئناف: الموجود يُتخطى                              #
# ============================================================ #
import os

workers = int(CFG["PREP_WORKERS"] or 0) or min(16, os.cpu_count() or 4)
sh([VENV_PY, PIPE, "prep",
    "--root", ROOT,
    "--out", DATA_DIR,
    "--workers", workers])

n = sum(1 for _ in DATA_DIR.rglob("*.ogg"))
print(f"\nمقاطع التدريب الجاهزة: {n:,} ملف Opus في {DATA_DIR}")
assert n >= 50, f"مقاطع قليلة جداً ({n}) — تأكد أن الخليتين 7 و8 أنزلتا بيانات ثم أعد المحاولة"
print("[جاهز] البيانات مبنية — تابع إلى الخلية 11 (تنزيل نقاط النموذج).")

In [ ]:
# ============================================================ #
#  الخلية ١١ — تنزيل نقاط النموذج الأساسية مسبقاً (≈2.2GB)      #
#  DiT_uvit_tat_xlsr_ema + XLSR-300m + HiFT + CAMPPlus +       #
#  OpenVoice — كلها في مجلد checkpoints داخل seed-vc           #
#  (يمنع تعطل بدء التدريب لاحقاً بسبب التنزيل، وقابل للاستئناف) #
# ============================================================ #
sh([VENV_PY, PIPE, "prefetch", "--repo", SEEDVC])
import shutil
ck = SEEDVC / "checkpoints"
total = sum(f.stat().st_size for f in ck.rglob("*") if f.is_file()) / 1e9 if ck.exists() else 0
print(f"\nحجم مخزون النقاط: {total:.2f}GB في {ck}")
print("[جاهز] كل شيء مطلوب للتدريب موجود — تابع إلى مرحلة التدريب.")

<div dir="rtl">

## مرحلة التدريب — كيف تعمل؟

- **نقطة البداية دائماً** الأوزان المنشورة `DiT_uvit_tat_xlsr_ema.pth` من `Plachta/Seed-VC` (لا تدريب
  من الصفر أبداً). إن وُجدت نقطة سابقة في مجلد التشغيل يُستأنف منها تلقائياً (يُحتفظ بآخر نقطتين
  `DiT_epoch_*_step_*.pth` فقط لتوفير المساحة).
- **السجل**: سطر `epoch E, step S, loss: L` كل 10 خطوات؛ نقطة حفظ كل `SAVE_EVERY` خطوة؛ وفي النهاية
  يُكتب `ft_model.pth` (أوزان DiT المضبوطة — وهي كل ما يلزم للنشر والاستدلال).
- **الإعدادات الموصى بها من دليل المستودع (FINETUNING.md)**: دفعة 2 لذاكرة 16GB و4 لـ 24GB؛
  200–500 خطوة لتحسين متحدث هدف واحد؛ 1500–4000 خطوة لتكيّف عام مع اللهجة؛ معدل التعلم مثبّت
  في الإعداد الرسمي — لا ترفعه لنموذج بحجم 25M باراميتر.
- **لماذا التدريب في الخلفية؟** يُطلق كعملية مستقلة (جلسة خاصة) تكتب في `train_console.log` — لو
  أغلقت المتصفح أو انقطع الإنترنت أو أُعيد تشغيل نواة الدفتر، **التدريب يتابع عمله**. راقبه بالخلية 13.
- **انقطاع Spot؟** أعد تشغيل الحاوية، افتح الدفتر، شغّل الخلايا بالترتيب — سيتخطى المنجز ويستأنف
  التدريب من آخر نقطة حفظ.

مدة تقديرية: 3000 خطوة على RTX 3090/4090/A10G في حدود **1–3 ساعات تقريباً** (حسب أطوال المقاطع)؛
على T4 أبطأ بوضوح. راقب معدل الخطوات في خلية المراقبة بعد أول دقيقة لتقدير زمنك الفعلي بدقة.

</div>

In [ ]:
# ============================================================ #
#  الخلية ١٢ — إطلاق التدريب (عملية خلفية مستقلة عن النواة)    #
#  لا تتأثر بإغلاق المتصفح أو انقطاعك أو إعادة تشغيل النواة    #
#  التقدم يُكتب في: runs/<run>/train_console.log               #
# ============================================================ #
import subprocess, os, time
from pathlib import Path

LOG = RUN_DIR / "train_console.log"
PIDF = RUN_DIR / "train.pid"
FT = RUN_DIR / "ft_model.pth"
CFG_TINY = SEEDVC / "configs" / "presets" / "config_dit_mel_seed_uvit_xlsr_tiny.yml"

def train_alive():
    if not PIDF.exists():
        return False
    try:
        os.kill(int(PIDF.read_text().strip()), 0)
        return True
    except (ValueError, ProcessLookupError, PermissionError):
        return False

BATCH = int(CFG.get("_BATCH") or 2)
MAXS = int(CFG["STEPS"])

if FT.exists():
    print(f"النموذج النهائي موجود أصلاً: {FT}")
    print("لتدريب جديد كلياً: غيّر RUN_NAME في الخلية 1 (أو احذف ft_model.pth ثم أعد هذه الخلية).")
elif train_alive():
    print(f"التدريب يعمل الآن (PID {PIDF.read_text().strip()}) — راقبه بالخلية 13.")
else:
    RUN_DIR.mkdir(parents=True, exist_ok=True)
    if PIDF.exists():
        PIDF.unlink()   # بقايا عملية ميتة
    cmd = [str(VENV_PY), "-u", "train.py",
           "--config", str(CFG_TINY),
           "--dataset-dir", str(DATA_DIR),
           "--run-name", CFG["RUN_NAME"],
           "--batch-size", str(BATCH),
           "--max-steps", str(MAXS),
           "--max-epochs", "1000",
           "--save-every", str(int(CFG["SAVE_EVERY"])),
           "--num-workers", str(int(CFG["TRAIN_WORKERS"])),
           "--gpu", "0"]
    print("أمر التدريب:")
    print("  cd " + str(SEEDVC) + " && " + " ".join(cmd) + "\n")
    log_fh = open(LOG, "a")
    proc = subprocess.Popen(cmd, cwd=str(SEEDVC), env=ENV,
                            stdout=log_fh, stderr=subprocess.STDOUT,
                            start_new_session=True)   # يستقل عن النواة نهائياً
    PIDF.write_text(str(proc.pid))
    time.sleep(3)
    if proc.poll() not in (None, 0):
        tail = LOG.read_text(errors="replace")[-3000:]
        raise RuntimeError("ماتت عملية التدريب فوراً! آخر السجل:\n" + tail)
    print(f"[انطلق] التدريب يعمل الآن في الخلفية — PID {proc.pid}")
    print(f"  السجل الكامل : {LOG}")
    print(f"  النقاط        : {RUN_DIR}")
    print(f"  الهدف         : {MAXS} خطوة (دفعة {BATCH}، حفظ كل {int(CFG['SAVE_EVERY'])})")
    print("\nالخطوة التالية: شغّل خلية «مراقبة التدريب» (13) الآن، وأعد تشغيلها كل فترة.")
    print("ويمكنك إغلاق المتصفح مطمئناً — التدريب مستقل عنه تماماً.")

In [ ]:
# ============================================================ #
#  الخلية ١٣ — مراقبة التدريب (أعد تشغيلها متى شئت)            #
#  تقدم + معدل + زمن متبقٍ + منحنى الخسارة + آخر السجل          #
#  (تسميات الرسم بالإنجليزية لأن matplotlib لا يشكّل            #
#   الحروف العربية المتصلة بشكل صحيح)                          #
# ============================================================ #
import re, os, time
from pathlib import Path
from IPython.display import display, Image as IPyImage

LOG = RUN_DIR / "train_console.log"
PIDF = RUN_DIR / "train.pid"
FT = RUN_DIR / "ft_model.pth"
STEP_RE = re.compile(r"epoch (\d+), step (\d+), loss: ([0-9.eE+-]+)")
MAXS = int(CFG["STEPS"])

def _alive():
    try:
        if PIDF.exists():
            os.kill(int(PIDF.read_text().strip()), 0)
            return True
    except Exception:
        pass
    return False

lines = LOG.read_text(errors="replace").splitlines() if LOG.exists() else []
pts = []
for ln in lines:
    m = STEP_RE.search(ln)
    if m:
        pts.append((int(m.group(2)), float(m.group(3))))

if FT.exists():
    status = "منجز — ft_model.pth موجود (انتقل إلى خلية التحقق 15)"
elif _alive():
    status = "يعمل الآن"
elif pts:
    status = "متوقف — العملية غير حية ولا يوجد نموذج نهائي (راجع آخر السجل)"
else:
    status = "لم يبدأ بعد"

steps = pts[-1][0] if pts else 0
start = PIDF.stat().st_mtime if PIDF.exists() else (LOG.stat().st_mtime if LOG.exists() else time.time())
elapsed = max(time.time() - start, 1.0)
rate = steps / elapsed if steps else 0.0
eta_min = (MAXS - steps) / rate / 60 if rate > 0 else float("nan")
pct = min(100.0 * steps / MAXS, 100.0) if MAXS else 0.0
filled = int(30 * pct / 100)
bar = "#" * filled + "-" * (30 - filled)

print(f"الحالة: {status}")
print(f"[{bar}] {pct:5.1f}%   step {steps}/{MAXS}   معدل {rate:.2f} خطوة/ث   زمن منقضٍ {elapsed/60:.0f} د   متبقٍ ≈ {eta_min:.0f} د")
if pts:
    print(f"آخر خسارة (EMA): {pts[-1][1]:.4f} عند الخطوة {steps}")

# ---- منحنى الخسارة ----
if len(pts) >= 2:
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    xs, ys = zip(*pts)
    fig, ax = plt.subplots(figsize=(9, 3.2), constrained_layout=True)
    ax.plot(xs, ys, lw=1.3, color="#d62728")
    ax.set_xlabel("step")
    ax.set_ylabel("EMA loss")
    ax.set_title(f"loss curve - {CFG['RUN_NAME']} ({len(pts)} points)")
    ax.grid(alpha=0.3)
    chart = RUN_DIR / "loss_curve.png"
    fig.savefig(chart, dpi=110)
    plt.close(fig)
    display(IPyImage(filename=str(chart)))
    print(f"(حُفظ أيضاً بشكل دائم: {chart})")

# ---- آخر السجل ----
print("\n— آخر 12 سطراً من train_console.log —")
for ln in lines[-12:]:
    print("  " + ln)
ckpts = sorted(RUN_DIR.glob("DiT_epoch_*_step_*.pth"), key=lambda p: int(p.stem.rsplit("_", 1)[-1]))
if ckpts:
    print(f"\nنقاط التفتيش المحفوظة (آخر 3): " + ", ".join(c.name for c in ckpts[-3:]))

In [ ]:
# ============================================================ #
#  الخلية ١٤ — (اختيارية) انتظار اكتمال التدريب                #
#  مراقبة حية تلقائية كل 30 ثانية حتى النهاية                  #
#  (بدلها: شغّل الخلية 13 يدوياً كل فترة — نفس الغرض)           #
# ============================================================ #
import re, os, time
from pathlib import Path
from IPython.display import clear_output

LOG = RUN_DIR / "train_console.log"
PIDF = RUN_DIR / "train.pid"
FT = RUN_DIR / "ft_model.pth"
STEP_RE = re.compile(r"epoch (\d+), step (\d+), loss: ([0-9.eE+-]+)")
MAXS = int(CFG["STEPS"])

def _alive():
    try:
        if PIDF.exists():
            os.kill(int(PIDF.read_text().strip()), 0)
            return True
    except Exception:
        pass
    return False

deadline = time.time() + 24 * 3600   # مهلة قصوى 24 ساعة
while time.time() < deadline:
    clear_output(wait=True)
    lines = LOG.read_text(errors="replace").splitlines() if LOG.exists() else []
    steps, loss = 0, None
    for ln in lines:
        m = STEP_RE.search(ln)
        if m:
            steps, loss = int(m.group(2)), float(m.group(3))
    pct = min(100.0 * steps / MAXS, 100.0)
    bar = "#" * int(30 * pct / 100)
    msg = f"[{bar:<30}] {pct:5.1f}%   step {steps}/{MAXS}"
    if loss is not None:
        msg += f"   loss {loss:.4f}"
    print(msg, flush=True)
    if FT.exists():
        print("\nاكتمل التدريب — ft_model.pth جاهز. انتقل إلى خلية التحقق (15).")
        break
    if not _alive():
        print("\nتوقفت عملية التدريب دون إنتاج النموذج النهائي.")
        print("شغّل خلية المراقبة (13) لرؤية آخر السجل — غالباً يُستأنف بإعادة الخلية 12.")
        break
    time.sleep(30)
else:
    print("انتهت مهلة الانتظار (24 ساعة) — استخدم الخلية 13 للفحص اليدوي.")

<div dir="rtl">

## التحقق والتصدير

قبل الاحتفال بالنموذج نجري **استدلالاً حقيقياً** بالنقطة الناتجة: مقطع مصدر من أمثلة Seed-VC يُحوَّل
بصوت مرجعي عراقي من بياناتك المجهزة، وتستمع للناتج داخل الدفتر مباشرة. بعدها نبني **حزمة التصدير**:

```
exports/iraqi_v1/
├── seed_uvit_tat_xlsr_tiny_iraqi.pth    الأوزان المضبوطة (للنشر المباشر)
├── config_dit_mel_seed_uvit_xlsr_tiny.yml
├── training_report.json                  تقرير التدريب (البيانات والكتالوج والتاريخ)
├── _stats.json                           ساعات/ملفات كل مصدر
├── train_console.log                     سجل التدريب كاملاً
├── loss_curve.png                        منحنى الخسارة
└── README.md                             أوامر الاستخدام الجاهزة
```

</div>

In [ ]:
# ============================================================ #
#  الخلية ١٥ — التحقق: استدلال حقيقي بالنقطة المضبوطة          #
#  (مصدر من أمثلة Seed-VC + مرجع عراقي من بياناتك -> تشغيل)    #
# ============================================================ #
from IPython.display import Audio, display
from pathlib import Path

FT = RUN_DIR / "ft_model.pth"
assert FT.exists(), "لا يوجد ft_model.pth — شغّل خلية الإطلاق (12) وراقب حتى الاكتمال"
CFG_TINY = SEEDVC / "configs" / "presets" / "config_dit_mel_seed_uvit_xlsr_tiny.yml"

sh([VENV_PY, PIPE, "verify",
    "--repo", SEEDVC,
    "--run-dir", RUN_DIR,
    "--ckpt", FT,
    "--config", CFG_TINY,
    "--data-dir", DATA_DIR], check=False)   # فشل verify ليس قاتلاً — الأوزان تصدَّر على أي حال

outs = sorted((RUN_DIR / "verify_out").glob("vc_*.wav"))
if outs:
    print(f"\nناتج التحويل: {outs[0].name}  ({outs[0].stat().st_size/1e6:.1f} MB)")
    print("استمع للناتج (المصدر إنجليزي يحمل الطابع الصوتي العراقي المرجعي):")
    display(Audio(filename=str(outs[0])))
    for extra in outs[1:3]:
        print(f"ناتج إضافي: {extra.name}")
        display(Audio(filename=str(extra)))
else:
    print("لم يُنتج ملف صوتي — راجع السجل أعلاه. (غير قاتل: الأوزان ستصدَّر في الخلية 16 على أي حال)")

In [ ]:
# ============================================================ #
#  الخلية ١٦ — تصدير الحزمة النهائية                           #
#  أوزان + إعداد + تقرير + سجلات — في مجلد ثابت على القرص      #
#  الدائم: exports/<اسم-التشغيل>/                              #
# ============================================================ #
import shutil
from pathlib import Path

FT = RUN_DIR / "ft_model.pth"
assert FT.exists(), "لا يوجد ft_model.pth — أكمل التدريب أولاً"
EXPORT_DIR = EXPORTS / CFG["RUN_NAME"]
CFG_TINY = SEEDVC / "configs" / "presets" / "config_dit_mel_seed_uvit_xlsr_tiny.yml"

sh([VENV_PY, PIPE, "export",
    "--run-dir", RUN_DIR,
    "--config", CFG_TINY,
    "--data-dir", DATA_DIR,
    "--out", EXPORT_DIR])

# توثيق إضافي يُنسخ مع الحزمة (قابلية إعادة الإنتاج جزء من الجودة)
for extra in (RUN_DIR / "train_console.log", RUN_DIR / "loss_curve.png",
              RUN_DIR / "config_dit_mel_seed_uvit_xlsr_tiny.yml", DATA_DIR / "_stats.json"):
    if extra.exists():
        shutil.copy2(extra, EXPORT_DIR / extra.name)
        print(f"  + {extra.name}")

total = 0
print(f"\nحزمة التصدير: {EXPORT_DIR}")
for p in sorted(EXPORT_DIR.rglob("*")):
    if p.is_file():
        s = p.stat().st_size
        total += s
        print(f"  {p.name:<46} {s/1e6:9.2f} MB")
print(f"  {'الإجمالي':<46} {total/1e6:9.2f} MB")

<div dir="rtl">

## الحفظ الدائم — ثلاث طبقات (حتى تنزّلها بعد سنة)

| الطبقة | أين تبقى | كم تدوم | كيف تنزّلها لاحقاً |
|---|---|---|---|
| 1. القرص الشبكي RunPod | `/workspace/iraqi_seedvc/` (Network Volume) | دائمة ما لم تحذف الـ Volume بنفسك | وصّل أي حاوية جديدة بنفس الـ Volume — أو نزّل عبر Jupyter / scp |
| 2. أرشيف tar.gz | `exports/rafidain_seedvc_<run>.tar.gz` | مع الطبقة 1 | ملف واحد: متصفح Jupyter → كليك يمين → Download، أو scp |
| 3. **HuggingFace Hub** | مستودع في حسابك (خاص افتراضياً) | **دائمة مجاناً في حسابك — لا علاقة لها بـ RunPod إطلاقاً** | `hf download <repo>` أو زر التنزيل من صفحة المستودع |

> **الوصية**: الطبقة 3 هي ضمانتك الحقيقية للمدى البعيد — حتى لو انتهى حسابك في RunPod أو حُذف
> القرص الشبكي، يبقى النموذج في HuggingFace قابلاً للتنزيل من أي مكان في العالم. تحتاج فقط رمز API
> بصلاحية كتابة من `hf.co/settings/tokens` تضعه في الخلية 1 ثم تشغّل الخلية 18.

كل ما أنتجته الرحلة يبقى أيضاً على القرص الدائم: البيانات الخام في `raw/`، وبيانات التدريب في
`data/iraqi_finetune/`، ونقاط التدريب والسجلات في `seed-vc/runs/<run>/` — أعد فتح الدفتر متى شئت
وستجد كل شيء في مكانه (وتعمل أي مرحلة من جديد إن احتجت).

</div>

In [ ]:
# ============================================================ #
#  الخلية ١٧ — أرشيف tar.gz واحد جاهز للتنزيل                  #
# ============================================================ #
import shutil
from pathlib import Path

EXPORT_DIR = EXPORTS / CFG["RUN_NAME"]
assert (EXPORT_DIR / "seed_uvit_tat_xlsr_tiny_iraqi.pth").exists(), "شغّل خلية التصدير (16) أولاً"

base = EXPORTS / f"rafidain_seedvc_{CFG['RUN_NAME']}"
archive = shutil.make_archive(str(base), "gztar", root_dir=str(EXPORTS), base_dir=EXPORT_DIR.name)
size_mb = Path(archive).stat().st_size / 1e6
print(f"الأرشيف جاهز: {archive}  ({size_mb:.1f} MB)")
print(f'''
لتنزيله الآن:
  • من Jupyter: لوحة الملفات (يساراً) → مجلد exports → كليك يمين على الملف → Download
  • أو من جهازك عبر SSH (تجد الأمر في لوحة RunPod → Connect → SSH over TLS):
      scp -P <المنفذ> root@<العنوان>:{archive} .
''')

In [ ]:
# ============================================================ #
#  الخلية ١٨ — الرفع الدائم إلى HuggingFace Hub                #
#  النموذج يُخزَّن في حسابك أنت، مجاناً، بلا علاقة بـ RunPod     #
#  (أعد تشغيلها في أي وقت مستقبلاً — ولو بعد شهور)              #
# ============================================================ #
from pathlib import Path

EXPORT_DIR = EXPORTS / CFG["RUN_NAME"]
HF_SCRIPT = ROOT / "hf_upload.py"
HF_SCRIPT.write_text(r'''
import os, time
from huggingface_hub import HfApi

token = os.environ["HF_TOKEN"]
export_dir = os.environ["NB_EXPORT_DIR"]
api = HfApi(token=token)
user = api.whoami()["name"]
repo_id = os.environ.get("NB_REPO_ID", "").strip() or f"{user}/rafidain-seedvc-iraqi"
private = os.environ.get("NB_PRIVATE", "1") == "1"
api.create_repo(repo_id, private=private, exist_ok=True)
print(f"[hf] رفع {export_dir} إلى {repo_id} (خاص = {private}) ...")
api.upload_folder(
    folder_path=export_dir,
    repo_id=repo_id,
    repo_type="model",
    commit_message=f"Rafidain Iraqi fine-tune ({time.strftime('%Y-%m-%d %H:%M')})",
)
print("[hf] REPO_URL: https://huggingface.co/" + repo_id)
''', encoding="utf-8")

if not CFG["HF_TOKEN"]:
    print("لا يوجد رمز HuggingFace بعد — لا مشكلة: كل أصولك محفوظة على القرص الشبكي،")
    print("ويمكنك تنفيذ الرفع الدائم في أي وقت لاحق (الخلية تعمل مستقلة):\n")
    print("  1) أنشئ حساباً مجانياً في huggingface.co إن لم يكن لديك")
    print("  2) من hf.co/settings/tokens أنشئ New Token بصلاحية Write وانسخه")
    print('  3) في الخلية 1 اضبط:  CFG["HF_TOKEN"] = "hf_ضع-الرمز-هنا"  ثم أعد تشغيلها')
    print("  4) أعد تشغيل هذه الخلية — سترفع الحزمة كاملة إلى مستودع خاص في حسابك")
else:
    assert (EXPORT_DIR / "seed_uvit_tat_xlsr_tiny_iraqi.pth").exists(), "شغّل خلية التصدير (16) أولاً"
    env = dict(ENV)
    env["HF_TOKEN"] = str(CFG["HF_TOKEN"])
    env["NB_EXPORT_DIR"] = str(EXPORT_DIR)
    env["NB_REPO_ID"] = str(CFG.get("HF_REPO_ID", ""))
    env["NB_PRIVATE"] = "1" if CFG["HF_PRIVATE"] else "0"
    sh([VENV_PY, HF_SCRIPT], env=env)
    print('''
[تم] النموذج الآن في حسابك على HuggingFace — بشكل دائم.
لتنزيله في أي وقت مستقبلاً من أي جهاز (بلا RunPod):
    pip install -U "huggingface_hub[cli]"
    hf download <اسم-حسابك>/rafidain-seedvc-iraqi --local-dir ./mymodel
أو من بايثون:
    from huggingface_hub import snapshot_download
    snapshot_download("<اسم-حسابك>/rafidain-seedvc-iraqi", local_dir="./mymodel")
أو مباشرة من صفحة المستودع في المتصفح.''')

In [ ]:
# ============================================================ #
#  الخلية ١٩ — (اختيارية) طبقة رابعة: rclone سحابي             #
#  Google Drive / S3 / Dropbox / OneDrive ...                  #
#  خلية مرجعية — نفّذ الأوامر يدوياً وقت الحاجة                 #
# ============================================================ #

# 1) التثبيت (مرة واحدة على الحاوية):
#      apt-get install -y rclone          # أو أحدث إصدار من rclone.org/install
#
# 2) الإعداد التفاعلي (مرة واحدة — يربط حسابك السحابي):
#      rclone config                      # أنشئ remote باسم مثلاً: mydrive
#      (ملف الإعداد يبقى في /root/.config/rclone/rclone.conf — انسخه إلى
#       /workspace/iraqi_seedvc/ ليبقى مع القرص الدائم بين الحاويات)
#
# 3) رفع حزمة النموذج:
#      rclone copy /workspace/iraqi_seedvc/exports/{run} mydrive:SeedVC_Iraqi/ -P
#
# 4) تنزيله لاحقاً من أي جهاز آخر فيه rclone مضبوط على الحساب نفسه:
#      rclone copy mydrive:SeedVC_Iraqi ./mymodel -P

print("خلية مرجعية — الأوامر جاهزة في التعليقات أعلاه.")
print("الطبقة الدائمة الأساسية تبقى HuggingFace Hub (الخلية 18) — الأسهل والأضمن.")

In [ ]:
# ============================================================ #
#  الخلية ٢٠ — (اختيارية) تحرير مساحة القرص                    #
#  تحذف المصادر الخام القابلة لإعادة الجلب فقط:                #
#    raw/yodas_webm + مخابئ parquet — وتبقى بيانات التدريب     #
#    المجهزة وكل النقاط والحزم كما هي                          #
#  ملاحظة: علامات .done تبقى، فلن يُعاد التنزيل تلقائياً؛       #
#  لإعادة الجلب مستقبلاً احذف ملف .done المناسب من raw/.done/  #
# ============================================================ #
import shutil
from pathlib import Path

UV = globals().get("UV") or shutil.which("uv")

def dir_gb(p: Path) -> float:
    if not p.exists():
        return 0.0
    return sum(f.stat().st_size for f in p.rglob("*") if f.is_file()) / 1e9

targets = [ROOT / "raw" / "yodas_webm", ROOT / "raw" / "_parquet", ROOT / "raw" / "_safaa_src"]
print(f"المساحة الحرة الآن: {shutil.disk_usage(ROOT).free/1e9:.1f} GB")
for t in targets:
    s = dir_gb(t)
    print(f"  {t.name:<12} {s:6.2f} GB" + ("   (قابل لإعادة الجلب بالخليتين 7 و8)" if s > 0 else "   (فارغ)"))

if CFG["CLEAN_RAW_AFTER"]:
    for t in targets:
        if t.exists():
            shutil.rmtree(t)
            print(f"  حُذف: {t.name}")
    if UV:
        sh([UV, "cache", "clean"], check=False)
    else:
        sh([str(VENV_PY), "-m", "pip", "cache", "purge"], check=False)
    print(f"المساحة الحرة بعد التنظيف: {shutil.disk_usage(ROOT).free/1e9:.1f} GB")
    print("بيانات التدريب الجاهزة في data/iraqi_finetune — لا حاجة للمصادر الخام إلا لإعادة التجهيز.")
else:
    print('\nللتفعيل: اجعل  CFG["CLEAN_RAW_AFTER"] = True  في الخلية 1 ثم أعد تشغيل هذه الخلية.')

In [ ]:
# ============================================================ #
#  الخلية ٢١ — النشر: كيف تستخدم النموذج المضبوط؟              #
#  (خلية إرشادية تطبع الأوامر الجاهزة بمساراتك الفعلية)        #
# ============================================================ #
from pathlib import Path

EXPORT_DIR = EXPORTS / CFG["RUN_NAME"]
W = EXPORT_DIR / "seed_uvit_tat_xlsr_tiny_iraqi.pth"
C = EXPORT_DIR / "config_dit_mel_seed_uvit_xlsr_tiny.yml"

if not W.exists():
    print("شغّل خلية التصدير (16) أولاً حتى تتوفر الحزمة.")
else:
    print(f'''
==========================================================
 نشر النموذج المضبوط — ثلاثة مسارات جاهزة
==========================================================

(أ) استدلال فوري في هذه الحاوية (البيئة جاهزة الآن):
    cd {SEEDVC}
    {VENV_PY} inference.py \\
        --checkpoint {W} \\
        --config     {C} \\
        --source <صوت-المتكلم-الأصلي.wav> \\
        --target  <صوت-مرجعي-عراقي.wav> \\
        --output  ./out

(ب) واجهة ويب Gradio للتجربة السريعة (تحتاج تثبيت gradio أولاً):
    uv pip install --python {VENV_PY} gradio   # أو: {VENV_PY} -m pip install gradio
    cd {SEEDVC}
    {VENV_PY} app_vc.py --checkpoint {W} --config {C}
    ثم افتح المنفذ 7860 (فعّل بروكسي HTTP من إعدادات الحاوية في لوحة RunPod)

(ج) خدمة رافِدَين الكاملة (REST + WebSocket لبث لحظي في المكالمات):
    المستودع منسوخ هنا: {RAFIDAIN}
    انسخ ما يلي في ملف .env الخاص بالخدمة (انظر .env.example داخل المستودع):

      RAFIDAIN__ENGINE__LOCAL_CHECKPOINT={W}
      RAFIDAIN__ENGINE__LOCAL_CONFIG={C}
      RAFIDAIN__ENGINE__DEVICE=cuda
      RAFIDAIN__API_KEYS=["<مفتاح-API-عشوائي-خاص-بك>"]

    التفاصيل الكاملة (Docker/GPU، الأمان، TLS، القياس):
      {RAFIDAIN}/DEPLOYMENT.md   و   {RAFIDAIN}/WEBUI.md

تذكير أخلاقي: استنساخ صوت شخص حقيقي يتطلب موافقته الصريحة الموثقة،
وأفصح للطرف الآخر عند ما يوجبه القانون — راجع قسم الأمان في README المستودع.
''')

<div dir="rtl">

## استكشاف الأخطاء

| المشكلة | الحل |
|---|---|
| `CUDA out of memory` أثناء التدريب | خفّض الدفعة: `CFG["BATCH"] = 2` (أو 1) في الخلية 1، أعد تشغيلها ثم أعد الخلية 12 — يستأنف من آخر نقطة |
| نفاد مساحة القرص | شغّل خلية التنظيف (20)، أو قلّل `YODAS_HOURS` وأعد الخليتين 8 و10 (الموجود يُتخطى) |
| انقطعت الحاوية (Spot) | أعد التشغيل وشغّل الخلايا بالترتيب — كل شيء يستأنف تلقائياً من آخر إنجاز |
| Rate limit من HuggingFace | اضبط `CFG["HF_ENDPOINT"] = "https://hf-mirror.com"` في الخلية 1 أو انتظر دقائق وأعد المحاولة |
| بطء تنزيل YODAS | ارفع `FETCH_WORKERS` إلى 32 في الخلية 1 ثم أعد الخلية 8 — يستأنف دون إعادة ما نُزّل |
| `too few training clips` في الخلية 10 | تأكد أن الخليتين 7 و8 أنزلتا بيانات (انظر مجلد `raw/`) ثم أعد المحاولة |
| فشلت خلية التحقق (15) | غير قاتلة — الأوزان تصدَّر على أي حال؛ جرّب أمر inference.py اليدوي المطبوع في الخلية 21 |
| أعدت تشغيل النواة وضاعت المتغيرات | شغّل الخلايا بالترتيب من 1 — الإعدادات محفوظة على القرص الدائم وتعود كما كانت |
| أريد التأكد أن التدريب حي | الخلية 13 تعرض حالة العملية وآخر الخطوات ومنحنى الخسارة |
| أريد تجربة نقطة وسيطة | ملفات `DiT_epoch_*_step_*.pth` في مجلد التشغيل تُقبل مباشرة في `inference.py --checkpoint` |

</div>

<div dir="rtl">

## الخلاصة — أين كل شيء؟ (خريطة المقتنيات الدائمة)

| الأصل | المسار الدائم (على القرص الشبكي) |
|---|---|
| حزمة النموذج النهائية | `/workspace/iraqi_seedvc/exports/<run>/` |
| أرشيف النموذج (ملف واحد) | `/workspace/iraqi_seedvc/exports/rafidain_seedvc_<run>.tar.gz` |
| نسخة HuggingFace الدائمة | `https://huggingface.co/<حسابك>/rafidain-seedvc-iraqi` (الخلية 18) |
| بيانات التدريب المجهزة | `/workspace/iraqi_seedvc/data/iraqi_finetune/` (Opus 24kHz + `_stats.json`) |
| نقاط التدريب والسجلات | `/workspace/iraqi_seedvc/seed-vc/runs/<run>/` (مع `loss_curve.png`) |
| البيئة القابلة لإعادة الاستخدام | `/workspace/iraqi_seedvc/venv/` (PyTorch 2.4.0 cu121 + كل الاعتماديات) |
| البيانات الخام (قابلة لإعادة الجلب) | `/workspace/iraqi_seedvc/raw/` |

**لتنزيل النموذج في أي وقت مستقبلاً** — الطبقة الأضمن (HuggingFace، بلا RunPod نهائياً):

```bash
pip install -U "huggingface_hub[cli]"
hf download <اسم-حسابك>/rafidain-seedvc-iraqi --local-dir ./mymodel
```

ثم تشغيله فوراً على أي جهاز فيه GPU:

```bash
git clone https://github.com/Plachtaa/seed-vc && cd seed-vc
python inference.py \
    --checkpoint ./mymodel/seed_uvit_tat_xlsr_tiny_iraqi.pth \
    --config     ./mymodel/config_dit_mel_seed_uvit_xlsr_tiny.yml \
    --source a.wav --target ref.wav --output ./out
```

أما إن رجعت إلى RunPod نفسها: وصّل حاوية جديدة بالقرص الشبكي ذاته وستجد كل شيء في `/workspace`
كما تركته تماماً — بيانات وبيئة ونقاطاً وحزماً.

</div>